# OFC Transformer: Aging · SHB · Unseen 평가

공식 OFC 데이터 저장소의 Train 데이터로 Transformer를 학습하고, 공개 TestGround의 정답을 이용해 Aging/SHB/Unseen 및 전체 대회식 점수를 계산합니다. **TestGround는 학습·모델 선택에 사용하지 않고 최종 사후 평가에만 사용합니다.** 이 점수는 비공개 Kaggle 리더보드 점수가 아닙니다.

In [ ]:
import subprocess, sys, torch
from pathlib import Path
print('Torch:',torch.__version__,'GPU:',torch.cuda.is_available())
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scikit-learn','matplotlib'],check=True)
# Colab 런타임에 git-lfs가 없으면 먼저 설치
subprocess.run(['bash','-lc','command -v git-lfs >/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs)'],check=True)
print('git-lfs:',subprocess.check_output(['git-lfs','version'],text=True).strip())
def sh(cmd):
    print('+',' '.join(map(str,cmd)),flush=True)
    subprocess.run(list(map(str,cmd)),check=True)

## 1. 코드·공식 데이터 가져오기

Git LFS CSV가 실제 파일로 내려받아졌는지 확인합니다. TestGround feature/label 파일이 저장소에 없으면 전처리 관리자 노트북을 먼저 실행해 생성해야 합니다.

In [ ]:
repo=Path('/content/LSCNS')
data=Path('/content/ofc-ml-challenge-data-code')
if not (repo/'.git').exists(): sh(['git','clone','--depth','1','https://github.com/kimheeseo/LSCNS.git',repo])
if not (data/'.git').exists(): sh(['bash','-lc',f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/ofc-ml-challenge/ofc-ml-challenge-data-code.git {data}'])
sh(['git','-C',data,'lfs','install','--local'])
sh(['git','-C',data,'lfs','pull'])
required=['Features/Train/train_features.csv','Features/Train/train_labels.csv','Features/TestGround/test_features.csv','Features/TestGround/test_labels.csv']
for rel in required:
    p=data/rel
    if not p.exists(): raise FileNotFoundError(f'{p} missing. Generate aggregate CSVs using official code/kaggle_feature_extraction_admin.ipynb.')
    with p.open('rb') as f: head=f.read(128)
    if head.startswith(b'version https://git-lfs.github.com/spec/v1'): raise RuntimeError(f'LFS pointer, not data: {p}; run git lfs pull')
print('Required official CSVs ready.')

## 2. 학습 설정

학습은 이 셀에서 시작합니다. 진행률은 데이터 준비 10%, 학습 10~90%, 최종 TestGround 평가·저장 90~100%로 표시합니다. 학습 epoch 도중 10% 구간을 통과할 때마다 출력됩니다.

In [ ]:
MODEL_DIR=repo/'26OFC_ML_challenge/PI_EDFA_Transformer'
OUTPUT=Path('/content/ofc_transformer_testground_results')
EPOCHS=250
PATIENCE=40
SEED=2026
print('Max epochs:',EPOCHS,'| patience:',PATIENCE)

In [ ]:
# Start official training split; TestGround is only evaluated after training is complete.
runner=MODEL_DIR/'transformer_edfa.py'
assert runner.exists(),runner
cmd=[sys.executable,str(runner),'--data-root',str(data),'--output',str(OUTPUT),
     '--epochs',str(EPOCHS),'--patience',str(PATIENCE),'--seed',str(SEED),
     '--evaluate-testground']
sh(cmd)

## 3. Aging · SHB · Unseen 결과 보기

점수는 낮을수록 좋습니다. 결과 JSON과 CSV, 샘플·채널 진단 그래프는 OUTPUT 폴더에 저장됩니다.

In [ ]:
import json, pandas as pd
from IPython.display import display
result=json.loads((OUTPUT/'testground_metrics.json').read_text())
rows=[]
for cat,metrics in result.items():
    rows.append({'Category':cat,**metrics})
display(pd.DataFrame(rows).round(6))
print('Results:',OUTPUT)
print('Files:',*[p.name for p in sorted(OUTPUT.iterdir())])

### 평가 해석
공식 TestGround는 공개된 사후 평가 데이터입니다. 학습 Train과 분포가 겹치거나 실제 대회 hidden split과 다를 수 있으므로 결과를 공식 우승팀의 현장 점수와 완전히 동등한 대결로 해석하지 않습니다.